# Load the PhiUSIIL dataset

Loads the raw PhiUSIIL CSV and checks it matches what UCI publishes: its shape and its label counts.

Get the file first by following the steps in [`ml/README.md`](../README.md). The notebook expects it at `ml/data/PhiUSIIL_Phishing_URL_Dataset.csv`.

**Safety:** the dataset holds real phishing links. They are only ever printed as plain text here. Never click or open one.

In [1]:
from pathlib import Path

import pandas as pd

CSV_NAME = "PhiUSIIL_Phishing_URL_Dataset.csv"
EXPECTED_ROWS = 235_795
EXPECTED_COLUMNS = 56
EXPECTED_LABEL_COUNTS = {1: 134_850, 0: 100_945}


def find_ml_dir(start: Path) -> Path:
    """Walk up to the repo root (the folder holding AGENTS.md), so the notebook runs from any working directory inside the repo."""
    for folder in [start, *start.parents]:
        if (folder / "AGENTS.md").exists() and (folder / "ml").is_dir():
            return folder / "ml"
    raise FileNotFoundError("Could not find the ml/ folder. Open this notebook from inside the LinkGuard repo.")


ML_DIR = find_ml_dir(Path.cwd().resolve())
DATA_PATH = ML_DIR / "data" / CSV_NAME

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}.\n"
        "Download it by following 'Get the dataset' in ml/README.md, then run this notebook again."
    )

print(f"Found {DATA_PATH.relative_to(ML_DIR.parent)}")

Found ml/data/PhiUSIIL_Phishing_URL_Dataset.csv


In [2]:
df = pd.read_csv(DATA_PATH)

rows, columns = df.shape
print(f"Rows:    {rows:,}")
print(f"Columns: {columns}")

if (rows, columns) != (EXPECTED_ROWS, EXPECTED_COLUMNS):
    raise ValueError(
        f"Expected {EXPECTED_ROWS:,} rows and {EXPECTED_COLUMNS} columns, got {rows:,} and {columns}. "
        "The file may be incomplete or a different version. Download it again using ml/README.md."
    )

Rows:    235,795
Columns: 56


## Label counts

In the **raw file**, `label` uses this coding:

| `label` | Meaning |
|---|---|
| `1` | legitimate |
| `0` | phishing |

This is the opposite of what LinkGuard will use. A later cleaning step recodes it so 1 means phishing.

In [3]:
label_counts = df["label"].value_counts().sort_index(ascending=False)
print(label_counts.rename({1: "1 (legitimate)", 0: "0 (phishing)"}).to_string())

actual_label_counts = {int(label): int(count) for label, count in label_counts.items()}
if actual_label_counts != EXPECTED_LABEL_COUNTS:
    raise ValueError(f"Expected label counts {EXPECTED_LABEL_COUNTS}, got {actual_label_counts}.")

print("\nShape and label counts match the published dataset.")

label
1 (legitimate)    134850
0 (phishing)      100945

Shape and label counts match the published dataset.


## A first look

A few rows, printed as plain text so no link can be clicked.

In [4]:
print(df[["URL", "label"]].head(5).to_string())

                                  URL  label
0    https://www.southbankmosaics.com      1
1            https://www.uni-mainz.de      1
2      https://www.voicefmradio.co.uk      1
3         https://www.sfnmjournal.com      1
4  https://www.rewildingargentina.org      1
